## Quickstart

This notebook mirrors the [Cecil quickstart](https://docs.cecil.earth/quickstart) end-to-end. By the time you reach the bottom you'll have:

- a Cecil API key set up in this environment,
- an Area of Interest (AOI),
- a subscription to a dataset on that AOI,
- an `xarray.Dataset` you can analyse and plot.

### 1. Install the SDK

Install the [`cecil`](https://pypi.org/project/cecil/) Python package. On Colab the `%pip` magic installs into the active kernel.

In [ ]:
%pip install -q cecil

### 2. Authenticate

Follow the [Cecil quickstart docs](https://docs.cecil.earth/quickstart) to obtain an API key. The SDK reads it from the `CECIL_API_KEY` environment variable.

On Colab the cell below will prompt you to paste the key — it stays in memory for the session and isn't persisted.

In [ ]:
import getpass
import os

if not os.environ.get("CECIL_API_KEY"):
    os.environ["CECIL_API_KEY"] = getpass.getpass("Cecil API key: ")

In [ ]:
from cecil import Client

client = Client()

### 3. Create an Area of Interest

An AOI is a reusable polygon you can subscribe many datasets to. The starter polygon below covers a slice of the [Gran Chaco](https://en.wikipedia.org/wiki/Gran_Chaco) in South America — swap in your own GeoJSON when you're ready.

`external_ref` is an optional string you can use to tag the AOI with an identifier from your own system.

In [ ]:
aoi = client.create_aoi(
    external_ref="Chaco Region",
    geometry={
        "type": "Polygon",
        "coordinates": [[
            [-60.530404375, -20.809574274],
            [-60.530404375, -21.173552046],
            [-59.99196906, -21.173552046],
            [-59.99196906, -20.809574274],
            [-60.530404375, -20.809574274],
        ]],
    },
)
aoi

### 4. Subscribe to a dataset

Browse [docs.cecil.earth/datasets](https://docs.cecil.earth/datasets) to find a dataset ID. The cell below uses the [Hansen Global Forest Change](https://glad.umd.edu/dataset/global-2010-tree-cover-30-m) dataset as an example — substitute any dataset ID you have access to.

Subscriptions are processed asynchronously: a few moments after you create one, the data is staged in object storage and ready to load.

In [ ]:
subscription = client.create_subscription(
    aoi_id=aoi.id,
    dataset_id="9659ec1d-7091-4f8b-9db5-e9fe07d2f508",  # Hansen Global Forest Change
)
subscription

In [ ]:
import time

from cecil.errors import HTTPError

# Subscriptions are processed asynchronously. While files are being staged
# in object storage, load_xarray may either raise an HTTPError or return
# an empty Dataset — we retry until variables are available.
while True:
    try:
        ds = client.load_xarray(subscription.id)
        if ds.data_vars:
            print("  status: ready")
            break
        print("  status: not ready yet — waiting…")
    except HTTPError as err:
        print(f"  status: not ready yet ({err.status_code}) — waiting…")
    time.sleep(10)

### 5. Load and analyse

For raster datasets, `client.load_xarray(...)` returns an [`xarray.Dataset`](https://docs.xarray.dev/en/stable/generated/xarray.Dataset.html) backed by lazy dask arrays. For vector datasets, use `client.load_dataframe(...)` to get a `pandas.DataFrame` instead.

In [ ]:
ds = client.load_xarray(subscription.id)
ds

In [ ]:
import matplotlib.pyplot as plt

# Pixel values above 0 indicate forest loss years.
forest_loss = ds["loss_year"].where(ds["loss_year"] > 0)

# Convert values from 1-24 to 2001-2024.
forest_loss_years = forest_loss + 2000

forest_loss_years.plot(cmap="plasma").colorbar.set_ticks(
    [2001, 2005, 2010, 2015, 2020, 2024]
)

plt.gca().set_title("Hansen Global Forest Change - Chaco Region")
plt.gca().set_aspect("equal")
plt.gca().set_facecolor("#101010")

plt.show()

### Next steps

- [Browse the dataset catalogue](https://docs.cecil.earth/datasets)
- [More worked examples](https://docs.cecil.earth/examples)
- [Full SDK reference](https://docs.cecil.earth/sdk)